# PRUEBAS

### Parámetros Comunes

In [1]:
import pandas as pd
import time
from datetime import datetime

from typing import Iterable, Optional, Tuple, Dict, Any
import datetime as dt

# Cargar configuración DINAMIDA de acuerdo al entorno
from dotenv import dotenv_values
import os
import sys

# Acceso a Datos
import psycopg2 as pg2
import pyodbc
import sqlalchemy
from sqlalchemy import text

# Determinar la ruta base del proyecto
# BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
BASE_DIR = "C:/ETL/FORECAST"  # Ruta fija para desarrollo local
CORE_DIR = os.path.join(BASE_DIR, 'forecast_core')
sys.path.insert(0, CORE_DIR)
print("Contenido de sys.path:")
for path in sys.path:
    print(path)

ENV_PATH = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")  # Toma Producción si está definido, o la ruta por defecto
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)
    
secrets = dotenv_values(ENV_PATH)
folder = f"{secrets['BASE_DIR']}/{secrets['FOLDER_DATOS']}"

# Verificar en que entorno está funcioando
print(f"Python executable: {sys.executable}")
print(f"PATH: {os.environ.get('PATH')}")


Contenido de sys.path:
C:/ETL/FORECAST\forecast_core
C:\Program Files\Python312\python312.zip
C:\Program Files\Python312\DLLs
C:\Program Files\Python312\Lib
C:\Program Files\Python312
c:\ETL\venv

c:\ETL\venv\Lib\site-packages
c:\ETL\venv\Lib\site-packages\win32
c:\ETL\venv\Lib\site-packages\win32\lib
c:\ETL\venv\Lib\site-packages\Pythonwin
Python executable: c:\ETL\venv\Scripts\python.exe
PATH: c:\ETL\venv\Scripts;C:\ETL\venv\Scripts;C:\Program Files\Python312\Scripts\;C:\Program Files\Python312\;C:\windows\system32;C:\windows;C:\windows\System32\Wbem;C:\windows\System32\WindowsPowerShell\v1.0\;C:\windows\System32\OpenSSH\;C:\Program Files\Git\cmd;C:\Program Files\HP\HP One Agent;C:\Program Files\Microsoft SQL Server\150\Tools\Binn\;C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\;C:\Program Files (x86)\Microsoft SQL Server\160\DTS\Binn\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\

In [ ]:
import os

print(os.getcwd())

# C:\ETL\FORECAST\forecast_core

## Funciones Comunes

In [5]:
#Solo importa lo necesario desde el módulo de funciones

def Close_Connection(conn): 
    if conn is not None:
        conn.close()
        # print("✅ Conexión cerrada.")    
    return True

def Open_Conn_Postgres():
    conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

def Open_Diarco_Postgres():
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

### Probar Nuevos Parámetros

In [6]:
def get_execution_execute_by_status(status):
    if not status:
        print("No hay estados para filtrar")
        return None
    
    conn = Open_Conn_Postgres()
    if conn is None:
        return None
    try:
        query = f"""
        SELECT   e.name, 
            m.method,
            fee.ext_supplier_code, 
            fee.last_execution,
            fee.supply_forecast_execution_status_id as fee_status_id,
            fee.timestamp,
            e.supply_forecast_model_id as forecast_model_id,
            fee.supply_forecast_execution_id as forecast_execution_id, 
            fee.id as forecast_execution_execute_id,
            fee.supply_forecast_execution_schedule_id as forecast_execution_schedule_id,
            e.supplier_id
            
        FROM public.spl_supply_forecast_execution_execute as fee
            LEFT JOIN  public.spl_supply_forecast_execution as e
                ON fee.supply_forecast_execution_id = e.id
            LEFT JOIN public.spl_supply_forecast_model as m
                ON e.supply_forecast_model_id= m.id

        WHERE fee.supply_forecast_execution_status_id = {status}
            AND fee.last_execution = true;
        """
        # Ejecutar la consulta SQL
        fexsts = pd.read_sql(query, conn) # type: ignore
        return fexsts
    except Exception as e:
        print(f"Error en get_execution_status: {e}")
        return None
    finally:
        Close_Connection(conn)       

def get_full_parameters(supply_forecast_model_id, execution_id): # Parametros del Modelo(default_value) y de la Ejecución 
    conn = Open_Conn_Postgres()
    if conn is None:
        return None
    try:
        cur = conn.cursor()
        ## Establecer orden específico de los parámetros
        query = """
            SELECT 
                mp.name, 
                mp.data_type, 
                mp.default_value, 
                ep.value, 
                mp.id,  
                mp.supply_forecast_model_id, 
                ep.supply_forecast_execution_id
            FROM public.spl_supply_forecast_model_parameter mp	
            FULL JOIN public.spl_supply_forecast_execution_parameter ep
                ON ep.supply_forecast_model_parameter_id = mp.id
            WHERE mp.supply_forecast_model_id = %s
                AND ep.supply_forecast_execution_id = %s
            ORDER BY 1;
        """

        # Ejecutar la consulta de manera segura
        cur.execute(query, (supply_forecast_model_id, execution_id))

        # Obtener los resultados
        result = cur.fetchall()

        # Convertir los resultados a un DataFrame de pandas
        columns = ["name", "data_type", "default_value", "value", "id", "supply_forecast_model_id", "supply_forecast_execution_id"]
        df_parameters = pd.DataFrame(result, columns=columns)

        cur.close()
        return df_parameters

    except Exception as e:
        print(f"Error en get_execution_parameter: {e}")
        return None
    finally:
        Close_Connection(conn)
                

In [21]:
import re
import math
import unicodedata

def _strip_prefix(name: str) -> str:
    """Descarta los 3 primeros caracteres si el patrón es NN_ o NN- ; luego normaliza a snakecase sin tildes."""
    if not isinstance(name, str):
        return ""
    s = name.strip()
    # Quitar prefijo NN_ o NN-
    if re.match(r"^\s*\d{2}[_-]", s):
        s = s[3:]
    # Normalizar tildes y espacios -> snakecase
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s

def _split_values(v) -> list[str]:
    """Convierte el 'value' string en lista de tokens (separadores: coma, punto y coma o pipe)."""
    if v is None:
        return []
    s = str(v).strip()
    if not s:
        return []
    parts = re.split(r"[,\|;]", s)
    return [p.strip() for p in parts if p.strip() != ""]

def _to_float_safe(tok: str):
    """Convierte token a float si parece número; admite decimal con '.' o ','."""
    t = tok.replace(",", ".")
    try:
        return float(t)
    except Exception:
        return None

def _to_int_list(vals: list[str]):
    out = []
    for t in vals:
        f = _to_float_safe(t)
        if f is not None:
            try:
                out.append(int(round(f)))
            except Exception:
                pass
    return out or None

def _first_float(vals: list[str], default=None):
    for t in vals:
        f = _to_float_safe(t)
        if f is not None and math.isfinite(f):
            return f
    return default

def _first_str(vals: list[str], default=None):
    for t in vals:
        if t != "":
            return t
    return default

def _extract_params_from_df(df_params: Optional[pd.DataFrame], algorithm: Optional[str] = None) -> Dict[str, Any]:
    """
    Nueva estrategia:
      - Ordena por 'name' ascendente (ya lo hace su SELECT) y procesa fila a fila.
      - Descarta prefijos NN_ y normaliza nombre.
      - Mapea valores separados por comas según nombre lógico + algoritmo.
      - Usa default_value si value viene nulo.
    Devuelve: {'ventana','f1','f2','f3','filtros','sucursales','rubros','subrubro1','subrubro2','subrubro3','fecha_base'}
    """
    out = {
        "ventana": 30,
        "f1": None, "f2": None, "f3": None,
        "filtros": None,
        "sucursales": None, "rubros": None,
        "subrubro1": None, "subrubro2": None, "subrubro3": None,
        "fecha_base": None,
    }
    if df_params is None or df_params.empty:
        return out

    cols = {c.lower(): c for c in df_params.columns}
    name_col = cols.get("name")
    val_col  = cols.get("value")
    def_col  = cols.get("default_value")

    # Procesar cada fila
    for _, r in df_params.iterrows():
        raw_name = r[name_col] if name_col else None
        key = _strip_prefix(raw_name if raw_name is not None else "")
        v   = None
        if val_col:
            v = r[val_col]
        if (v is None or str(v).strip() == "") and def_col:
            v = r[def_col]

        vals = _split_values(v)

        # --- Filtros (solo impactan generar_datos) ---
        if key == "sucursales":
            out["sucursales"] = _to_int_list(vals)
            continue
        if key == "rubros":
            out["rubros"] = _to_int_list(vals)
            continue
        if key == "subrubro1":
            out["subrubro1"] = _to_int_list(vals)
            continue
        if key == "subrubro2":
            out["subrubro2"] = _to_int_list(vals)
            continue
        if key == "subrubro3":
            out["subrubro3"] = _to_int_list(vals)
            continue

        # --- Comunes / base temporal ---
        if key in ("ventana", "period_length", "period_lengh", "dias", "n_dias"):
            n = _first_float(vals)
            if n is not None and n > 0:
                out["ventana"] = int(round(n))
            continue
        if key == "fecha_base":
            out["fecha_base"] = _first_str(vals)
            continue

        # --- Algoritmo-dependientes ---
        algo = (algorithm or "").strip().upper()

        # ALGO_01: factores ponderados
        if algo == "ALGO_01":
            if key in ("factores", "pesos", "weights"):
                # e.g., "0.8,0.1,0.2"
                f = [_to_float_safe(t) for t in vals]
                if len(f) >= 1 and out["f1"] is None: out["f1"] = f[0]
                if len(f) >= 2 and out["f2"] is None: out["f2"] = f[1]
                if len(f) >= 3 and out["f3"] is None: out["f3"] = f[2]
                continue
            if key in ("factor_last","peso_ultimo","w_last"):
                out["f1"] = _first_float(vals, out["f1"]); continue
            if key in ("factor_previous","peso_previo","w_prev"):
                out["f2"] = _first_float(vals, out["f2"]); continue
            if key in ("factor_year","peso_anio","w_year"):
                out["f3"] = _first_float(vals, out["f3"]); continue

        # ALGO_03: Holt-Winters
        if algo == "ALGO_03":
            if key in ("periodos","periods"):
                n = _first_float(vals)
                if n is not None and n > 0: out["f1"] = int(round(n))
                continue
            if key in ("estacionalidad","seasonal"):
                out["f2"] = _first_str(vals, out["f2"]); continue   # 'add' / 'mul'
            if key in ("tendencia","trend"):
                out["f3"] = _first_str(vals, out["f3"]); continue   # 'add' / 'mul'

        # ALGO_04: EWMA
        if algo == "ALGO_04":
            if key in ("alpha","alfa"):
                out["f1"] = _first_float(vals, out["f1"]); continue

        # ALGO_07: ventana base móvil x factor
        if algo == "ALGO_07":
            if key == "factor":
                out["f1"] = _first_float(vals, out["f1"]); continue
            # fecha_base ya se mapeó; por compat, si viniera aquí:
            if key in ("base","base_date"):
                out["fecha_base"] = _first_str(vals, out["fecha_base"]); continue

        # Si aparece un nombre no reconocido, se ignora (o podrían loguearlo).
        # print(f"[WARN] Parámetro no reconocido: {key} -> {vals}")

    return out

In [ ]:
fes = get_execution_execute_by_status(10)

if fes is None or fes.empty:
        print("⚠️ No hay ejecuciones con estado 10 (FORECAST PENDIENTE) para procesar.")
        sys.exit(0)

for _, row in fes[fes["fee_status_id"] == 10].iterrows():  # type: ignore
    algoritmo = row["name"]
    name = algoritmo.split("_ALGO")[0]
    method = row["method"]
    execution_id = row["forecast_execution_id"]
    id_proveedor = row["ext_supplier_code"]
    forecast_execution_execute_id = row["forecast_execution_execute_id"]
    supply_forecast_model_id = row["forecast_model_id"]

    try:
        id_proveedor = int(float(id_proveedor))
    except Exception:
        pass

    print(f"Procesando ejecución: {name} - Método: {method} - ID_Proveedor: {id_proveedor} - Argoritmo: {algoritmo}")
    print(f"Parámetros del modelo (ID={supply_forecast_model_id}) y de la ejecución (ID={execution_id})...")
   


    start_time = time.time()

    try:
        df_params = get_full_parameters(supply_forecast_model_id, execution_id)
        p = _extract_params_from_df(df_params, method)

        print(f"[PARAMS RAW] {df_params.to_dict(orient='records')}")
        print(f"[PARAMS EXTRACTED] {p}")
        # Parámetros específicos

        ventana = p.get("ventana", 30)
        f1 = p.get("f1")
        f2 = p.get("f2")
        f3 = p.get("f3")

        filtros_crudos = None  # ya no hace falta un JSON; usamos campos individuales
        sucursales = p.get("sucursales")
        rubros = p.get("rubros")
        subrubro1 = p.get("subrubro1")
        subrubro2 = p.get("subrubro2")
        subrubro3 = p.get("subrubro3")

        print(f"[PARAMS] ventana={ventana} | f1={f1} | f2={f2} | f3={f3}")
        # En ALGO_07, si enviaron fecha_base:
        if p.get("fecha_base") and f2 is None:
            f2 = p["fecha_base"]

        if filtros_crudos:
            print(f"[PARAMS] filtros(raw)={filtros_crudos}")
        if any([sucursales, rubros, subrubro1, subrubro2, subrubro3]):
            print(f"[PARAMS] overrides sucursales={sucursales} rubros={rubros} sr1={subrubro1} sr2={subrubro2} sr3={subrubro3}")

        # update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=15)

        # get_forecast(
        #     id_proveedor=id_proveedor,
        #     lbl_proveedor=name,
        #     period_lengh=ventana,
        #     algorithm=method,
        #     f1=f1,
        #     f2=f2,
        #     f3=f3,
        #     current_date=None,
        #     filtros=filtros_crudos,
        #     sucursales=sucursales,
        #     rubros=rubros,
        #     subrubro1=subrubro1,
        #     subrubro2=subrubro2,
        #     subrubro3=subrubro3,
        # )

        # update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=20)

        elapsed = round(time.time() - start_time, 2)
        print(f"✅ FORECAST : {algoritmo} procesado - Tiempo parcial: {elapsed} segundos")
        print("✅ Ejecución completada con éxito.")

    except Exception as e:
        print(f"❌ Error durante la ejecución del forecast: {e}")
        try:
            #update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=99)
            print("✅ Estado de ejecución actualizado a ERROR.")
        except Exception:
            pass




Procesando ejecución: 32291_PRUEBA_PARAM - Método: ALGO_07 - ID_Proveedor: 32291 - Argoritmo: 32291_PRUEBA_PARAM_ALGO_07
Parámetros del modelo (ID=33ad2395-3c46-4f73-b857-4b218f222acd) y de la ejecución (ID=9b3b2c65-f31b-4ca6-9947-6c503b0267e1)...
[PARAMS RAW] [{'name': '01_Ventana', 'data_type': 'int', 'default_value': '30', 'value': '30', 'id': 'c27bf7dc-2e58-4f56-9b6f-5727161f9691', 'supply_forecast_model_id': '33ad2395-3c46-4f73-b857-4b218f222acd', 'supply_forecast_execution_id': '9b3b2c65-f31b-4ca6-9947-6c503b0267e1'}, {'name': '02_Factor', 'data_type': 'float', 'default_value': '1.1', 'value': '1.1', 'id': 'a48a442d-1d69-4c58-9eac-141a90e7f7ca', 'supply_forecast_model_id': '33ad2395-3c46-4f73-b857-4b218f222acd', 'supply_forecast_execution_id': '9b3b2c65-f31b-4ca6-9947-6c503b0267e1'}, {'name': '03_Fecha_base', 'data_type': 'date', 'default_value': '2024-12-31', 'value': '2024-06-01', 'id': '9a2490a8-cea1-4560-9f89-3c4249500506', 'supply_forecast_model_id': '33ad2395-3c46-4f73-b857

C:\Users\eduar\AppData\Local\Temp\ipykernel_23764\949814323.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fexsts = pd.read_sql(query, conn) # type: ignore


In [ ]:
def extender_datos_forecast(algoritmo, name, id_proveedor):
    # Recuperar Historial de Ventas
    df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

    # Convertir tipos de datos    
    df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
    df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
    df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

    if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
        print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
        df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
        print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")
    
    # Recuperar Maestro de Artículos
    articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')
    
    # Recuperar Maestro de Artículos
    stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

    # Recuperando Forecast Calculado
    df_forecast = pd.read_csv(f'{folder}/{algoritmo}_Solicitudes_Compra.csv')
    df_forecast.fillna(0)   # Por si se filtró algún missing value
    print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")
    
    conn = Open_Conn_Postgres()
    
    # Obtener Sites
    stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
    stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
    stores['code'] = stores['code'].astype(int)

    # Obtener Productos
    products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
    products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
    products['ext_code'] = products['ext_code'].astype(int)
    assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


    Close_Connection(conn)

    # Unir con productos y validar
    df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
    df_merged.rename(columns={'id': 'product_id'}, inplace=True)
    df_merged.drop(columns=['ext_code', 'description'], inplace=True)

    # Unir con sites y validar
    df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
    df_merged.rename(columns={'id': 'site_id'}, inplace=True)
    df_merged.drop(columns=['code', 'name'], inplace=True)

    # Validación de integridad referencial
    errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
    if not errores.empty:
        print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
        errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
            f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
        raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

    
    df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
    # -- COMBINAR ARTÍCULOS y STOCK --
    df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
    df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
    df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
    df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

    duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
    if not duplicados.empty:
        print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
        duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

        #Elimino Duplicados
        df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

    df_merged = df_merged.merge(
        df_nuevo,
        left_on=['Sucursal', 'Codigo_Articulo'],
        right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
        how='left'
    )
    df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

    print(f"🔍 Forecast original: {len(df_forecast)} registros")
    print(f"📈 Forecast extendido: {len(df_merged)} registros")

    return df_merged

In [ ]:
name = "34229_Resistack"
algoritmo = "ALGO_05"
id_proveedor = 34229

In [ ]:
print(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')

## PRUEBA INICIAL de DATAFRAMES


In [ ]:
# Recuperar Historial de Ventas
df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

# Convertir tipos de datos    
df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
    print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
    df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
    print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")

In [ ]:
# Recuperar Maestro de Artículos
articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')

# Recuperar Maestro de Artículos
stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

# Recuperando Forecast Calculado
df_forecast = pd.read_csv(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')
df_forecast.fillna(0)   # Por si se filtró algún missing value
print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")

In [ ]:
# Completar con UUIDs de Connexa
conn = Open_Conn_Postgres()
    
# Obtener Sites
stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
stores['code'] = stores['code'].astype(int)

# Obtener Productos
products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
products['ext_code'] = products['ext_code'].astype(int)
assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


Close_Connection(conn)

In [ ]:
# Unir con productos y validar
df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
df_merged.rename(columns={'id': 'product_id'}, inplace=True)
df_merged.drop(columns=['ext_code', 'description'], inplace=True)


In [ ]:
# Unir con sites y validar
df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
df_merged.rename(columns={'id': 'site_id'}, inplace=True)
df_merged.drop(columns=['code', 'name'], inplace=True)

In [ ]:
# Validación de integridad referencial
errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
if not errores.empty:
    print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
    errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
        f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
    raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

In [ ]:
df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
# -- COMBINAR ARTÍCULOS y STOCK --
df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

In [ ]:
duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
if not duplicados.empty:
    print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
    duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

    #Elimino Duplicados
    df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

df_merged = df_merged.merge(
    df_nuevo,
    left_on=['Sucursal', 'Codigo_Articulo'],
    right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
    how='left'
)
df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

print(f"🔍 Forecast original: {len(df_forecast)} registros")
print(f"📈 Forecast extendido: {len(df_merged)} registros")


# NUEVA VERSIÓN
### PRUEBA SELECCIÓN DE SURTIDO y SUCURSALES






In [ ]:
# Funciones Utlilzadas

def Open_Connexa_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None   
    
def Open_Diarco_Data_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PG_USER']
    DB_PASS = secrets['PG_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PG_HOST']
    DB_PORT = secrets['PG_PORT']
    DB_NAME = secrets['PG_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None 
def get_forecast( id_proveedor, lbl_proveedor, period_lengh=30, algorithm='basic', f1=None, f2=None, f3=None, current_date=None, sucursales=None, rubros=None):
    """
    Genera la predicción de demanda según el algoritmo seleccionado.

    Parámetros:
    - id_proveedor: ID del proveedor.
    - lbl_proveedor: Etiqueta del proveedor.
    - period_lengh: Número de días del período a analizar (por defecto 30).
    - algorithm: Algoritmo a utilizar.
    - current_date: Fecha de referencia; si es None, se toma la fecha máxima de los datos.
    - factores de ponderación: F1, F2, F3  (No importa en que unidades estén, luego los hace relativos al total del peso)

    Retorna:
    - Un DataFrame con las predicciones.
    """
    
    print('Dentro del get_forecast LOCAL')
    print(f'FORECAST control: {id_proveedor} - {lbl_proveedor} - ventana: {period_lengh} - {algorithm} factores: {f1} - {f2} - {f3} ')
   
    # Generar los datos de entrada parampetricos
    if sucursales is not None:
        print(f'Filtrando por sucursales: {sucursales}')
        sucursales = ' IN (' + ','.join(map(str, sucursales)) + ')'
    else:
        sucursales = ' <> 0'
    if rubros is not None:
        print(f'Filtrando por rubros: {rubros}')
        rubros = ' IN (' + ','.join(f"'{r}'" for r in rubros) + ')'
    else:
        rubros = ' <> 0'
    
    data, articulos = generar_datos(id_proveedor, lbl_proveedor, period_lengh, sucursales, rubros) # type: ignore

    # Determinar la fecha base
    if current_date is None:
        current_date = data['Fecha'].max()  # type: ignore # Se toma la última fecha en los datos
    else:
        current_date = pd.to_datetime(current_date)  # Se asegura que sea un objeto datetime
    print(f'Fecha actual {current_date}')


def build_params(
    id_proveedor: int,
    desde: str | dt.date,
    sucursales: Optional[Iterable[int]] = None,
    rubros: Optional[Iterable[int]] = None
) -> Dict[str, Any]:
    """
    Devuelve el dict de parámetros en el formato esperado.
    Si sucursales o rubros vienen vacíos, los tratamos como NULL para no filtrar.
    """
    def _arr_or_null(x):
        if x is None:
            return None
        x = list(x)
        return x if len(x) > 0 else None

    print(f"[DEBUG] Parámetros construidos: id_proveedor={id_proveedor}, desde={desde}, sucursales={sucursales}, rubros={rubros}")
    return {
        "id_proveedor": int(id_proveedor),
        "desde": desde,                           # 'YYYY-MM-DD' o date
        "sucursales": _arr_or_null(sucursales),   # e.g. [1, 2, 3]  -> ANY(...)
        "rubros": _arr_or_null(rubros)            # e.g. [10, 20]   -> ANY(...)
    }


# PRUEBA DE NUEVA VERSIÖN

In [ ]:

import os
import hashlib
from datetime import datetime, timedelta
from typing import Iterable, Optional, Tuple, List, Union

import numpy as np
import pandas as pd

def Open_Diarco_Data(): 
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    #print (conn_str)
    for i in range(5):
        try:    
            conn = pg2.connect(conn_str)
            return conn
        except Exception as e:
            print(f'Error en la conexión: {e}')
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

# -*- coding: utf-8 -*-
# 2025-06-21: Filtros opcionales por sucursales, rubros y subrubros via join a src.m_3_articulos
import os
import hashlib
from datetime import datetime, timedelta
from typing import Iterable, Optional, Tuple, List, Union

import numpy as np
import pandas as pd

def generar_datos(
    id_proveedor: int,
    etiqueta: str,
    ventana: Optional[int],
    sucursales: Optional[Union[int, Iterable[int]]] = None,
    rubros: Optional[Union[int, Iterable[int]]] = None,
    subrubro1: Optional[Union[int, Iterable[int]]] = None,
    subrubro2: Optional[Union[int, Iterable[int]]] = None,
    subrubro3: Optional[Union[int, Iterable[int]]] = None,
):
    """
    Genera el dataset base de forecast para un proveedor, con filtros opcionales por sucursales, rubros y subrubros.

    - Proveedor (obligatorio).
    - Sucursales (opcional): int o lista de int.
    - Rubros (opcional): int o lista de int -> se evalúa contra m_3_articulos.c_rubro.
    - Subrubros (opcionales): c_subrubro_1 / c_subrubro_2 / c_subrubro_3.

    Retorna (data, articulos):
      * data: merge Artículos x Ventas (puede contener Unidades=0 cuando no hay ventas).
      * articulos: set de artículos vigentes por sucursal (ya filtrado).
    """

    # -------- Helpers internos --------
    def _to_int_list(val: Optional[Union[int, Iterable[int]]]) -> Optional[List[int]]:
        if val is None:
            return None
        if isinstance(val, (int, np.integer)):
            return [int(val)]
        try:
            lst = list(val)  # type: ignore
        except TypeError:
            return None
        out: List[int] = []
        for x in lst:
            if pd.isna(x):
                continue
            out.append(int(x))
        return out or None

    def _filters_signature(sucs, rubs, s1, s2, s3) -> str:
        base = (
            f"suc={','.join(map(str, sorted(sucs))) if sucs else 'ALL'}|"
            f"rub={','.join(map(str, sorted(rubs))) if rubs else 'ALL'}|"
            f"sr1={','.join(map(str, sorted(s1))) if s1 else 'ALL'}|"
            f"sr2={','.join(map(str, sorted(s2))) if s2 else 'ALL'}|"
            f"sr3={','.join(map(str, sorted(s3))) if s3 else 'ALL'}"
        )
        return hashlib.md5(base.encode('utf-8')).hexdigest()[:10]

    def _build_cte(join_m3: bool, cond_suc: bool, cond_rub: bool, cond_sr1: bool, cond_sr2: bool, cond_sr3: bool) -> str:
        """
        Arma el CTE articulos_filtrados, con join opcional a src.m_3_articulos cuando hay filtro por rubro/subrubros.
        """
        join_sql = "JOIN src.m_3_articulos m ON m.c_articulo = a.c_articulo\n" if join_m3 else ""
        cond_suc_sql  = "  AND a.c_sucu_empr = ANY(%(sucursales)s)\n"        if cond_suc  else ""
        cond_rub_sql  = "  AND m.c_rubro = ANY(%(rubros)s)\n"                if cond_rub  else ""
        cond_sr1_sql  = "  AND m.c_subrubro_1 = ANY(%(subrubro1)s)\n"        if cond_sr1  else ""
        cond_sr2_sql  = "  AND m.c_subrubro_2 = ANY(%(subrubro2)s)\n"        if cond_sr2  else ""
        cond_sr3_sql  = "  AND m.c_subrubro_3 = ANY(%(subrubro3)s)\n"        if cond_sr3  else ""

        # Nota: incluimos columnas de m_* para trazabilidad; si no las necesitan, pueden removerlas del SELECT.
        return f"""
            WITH articulos_filtrados AS (
                SELECT DISTINCT
                    a.c_sucu_empr, a.c_articulo, a.c_proveedor_primario, a.abastecimiento, a.cod_cd, a.habilitado,
                    a.cod_comprador, a.q_factor_compra, a.full_capacity_pallet, a.number_of_layers, a.number_of_boxes_per_layer,
                    {'m.c_rubro, m.c_subrubro_1, m.c_subrubro_2, m.c_subrubro_3,' if join_m3 else ''}
                    CURRENT_DATE AS fecha_extraccion_cte
                FROM src.base_productos_vigentes a
                {join_sql}WHERE a.c_proveedor_primario = %(id_proveedor)s
                AND a.habilitado = 1
            {cond_suc_sql}{cond_rub_sql}{cond_sr1_sql}{cond_sr2_sql}{cond_sr3_sql}
            )
            """.lstrip()

    # -------- Normalización de filtros y cache --------
    suc_list  = _to_int_list(sucursales)
    rub_list  = _to_int_list(rubros)
    sr1_list  = _to_int_list(subrubro1)
    sr2_list  = _to_int_list(subrubro2)
    sr3_list  = _to_int_list(subrubro3)

    sig = _filters_signature(suc_list, rub_list, sr1_list, sr2_list, sr3_list)

    folder =  os.path.join(BASE_DIR, secrets["FOLDER_DATOS"])
    suffix = f"__{sig}"

    archivo_datos      = f"{folder}/{etiqueta}{suffix}.csv"
    archivo_articulos  = f"{folder}/{etiqueta}{suffix}_articulos.csv"
    archivo_stock      = f"{folder}/{etiqueta}{suffix}_stock_sucursal.csv"
    archivo_demanda    = f"{folder}/{etiqueta}{suffix}_Demanda.csv"
    archivo_ventas_out = f"{folder}/{etiqueta}{suffix}_Ventas.csv"

    # -------- Cache del día --------
    if os.path.exists(archivo_datos):
        fecha_mod = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_mod.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=["Codigo_Articulo", "Sucursal", "Fecha"], how="all")
                data["Codigo_Articulo"] = pd.to_numeric(data["Codigo_Articulo"], errors="coerce").astype("Int64")
                data["Sucursal"]        = pd.to_numeric(data["Sucursal"], errors="coerce").astype("Int64")
                data["Fecha"]           = pd.to_datetime(data["Fecha"], errors="coerce")

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=["C_ARTICULO", "C_SUCU_EMPR"], how="all")
                print(f"-> Datos recuperados del CACHE: proveedor={id_proveedor}, etiqueta={etiqueta}, filtros={sig}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer cache: {e}")
        else:
            for f in (archivo_datos, archivo_articulos, archivo_stock, archivo_demanda, archivo_ventas_out):
                if os.path.exists(f):
                    try: os.remove(f)
                    except Exception: pass
            print(f"-> Cache obsoleto eliminado para filtros={sig}")

    # -------- Generación desde BD --------
    print(f"-> Generando datos para proveedor={id_proveedor}, etiqueta={etiqueta}, filtros={sig}")
    conn = Open_Diarco_Data()
    try:
        params = {"id_proveedor": int(id_proveedor)}
        if suc_list: params["sucursales"] = suc_list
        if rub_list: params["rubros"]     = rub_list
        if sr1_list: params["subrubro1"]  = sr1_list
        if sr2_list: params["subrubro2"]  = sr2_list
        if sr3_list: params["subrubro3"]  = sr3_list

        join_m3 = bool(rub_list or sr1_list or sr2_list or sr3_list)
        cte = _build_cte(
            join_m3=join_m3,
            cond_suc=bool(suc_list),
            cond_rub=bool(rub_list),
            cond_sr1=bool(sr1_list),
            cond_sr2=bool(sr2_list),
            cond_sr3=bool(sr3_list),
        )

        # --- ARTÍCULOS (ya filtrados) ---
        sql_articulos = f"""
            {cte}
            SELECT DISTINCT
                c_sucu_empr, c_articulo, c_proveedor_primario, abastecimiento, cod_cd, habilitado,
                cod_comprador AS c_comprador,
                q_factor_compra, full_capacity_pallet, number_of_layers, number_of_boxes_per_layer,
                {('c_rubro, c_subrubro_1, c_subrubro_2, c_subrubro_3,' if join_m3 else '')}
                fecha_extraccion_cte
            FROM articulos_filtrados
            ORDER BY c_articulo, c_proveedor_primario;
            """.strip()

        articulos = pd.read_sql(sql_articulos, conn, params=params)  # type: ignore
        if articulos.empty:
            print(f"❗ No se encontraron artículos para proveedor={id_proveedor} con los filtros indicados.")
            Close_Connection(conn)
            return None, None

        # Normalización y tipos
        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={"COD_COMPRADOR": "C_COMPRADOR"}, inplace=True)
        for col in [
            "C_SUCU_EMPR","C_ARTICULO","C_PROVEEDOR_PRIMARIO","ABASTECIMIENTO","HABILITADO",
            "C_COMPRADOR","Q_FACTOR_COMPRA","FULL_CAPACITY_PALLET","NUMBER_OF_LAYERS","NUMBER_OF_BOXES_PER_LAYER",
            "C_RUBRO","C_SUBRUBRO_1","C_SUBRUBRO_2","C_SUBRUBRO_3"
        ]:
            if col in articulos.columns:
                articulos[col] = pd.to_numeric(articulos[col], errors="coerce").astype("Int64")

        print(archivo_articulos)

        articulos.to_csv(archivo_articulos, index=False, encoding="utf-8")
        print(f"---> Datos de Artículos guardados")

        # --- STOCK x SUCURSAL (filtrado por el CTE) ---
        sql_stock = f"""
            {cte}
            SELECT
                s.codigo_articulo, s.codigo_sucursal, s.codigo_proveedor, s.pedido_sgm, s.stock,
                s.pedido_pendiente, s.i_lista_calculado, s.factor_venta, s.ultimo_ingreso, s.fecha_ultimo_ingreso,
                s.fecha_ultima_venta, s.precio_venta, s.precio_costo, s.q_dias_stock, s.transfer_pendiente,
                s.venta_unidades_1q, s.venta_unidades_2q
            FROM src.base_stock_sucursal s
            JOIN articulos_filtrados af
            ON af.c_articulo = s.codigo_articulo AND af.c_sucu_empr = s.codigo_sucursal
            WHERE s.codigo_proveedor = %(id_proveedor)s
            ORDER BY s.codigo_articulo, s.codigo_sucursal;
            """.strip()

        stock_sucursal = pd.read_sql(sql_stock, conn, params=params)  # type: ignore
        if stock_sucursal.empty:
            print(f"❗ No se encontraron registros en base_stock_sucursal para proveedor={id_proveedor} y filtros.")
        else:
            stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan).fillna(0)
            stock_sucursal.columns = stock_sucursal.columns.str.upper()

            for c in ["CODIGO_SUCURSAL", "CODIGO_ARTICULO", "CODIGO_PROVEEDOR", "FACTOR_VENTA"]:
                if c in stock_sucursal.columns:
                    stock_sucursal[c] = pd.to_numeric(stock_sucursal[c], errors="coerce").astype("Int64")

            for c in [
                "PEDIDO_SGM","STOCK","PEDIDO_PENDIENTE","I_LISTA_CALCULADO",
                "ULTIMO_INGRESO","PRECIO_VENTA","PRECIO_COSTO",
                "TRANSFER_PENDIENTE","VENTA_UNIDADES_1Q","VENTA_UNIDADES_2Q",
            ]:
                if c in stock_sucursal.columns:
                    stock_sucursal[c] = pd.to_numeric(stock_sucursal[c], errors="coerce").astype("Float64")

            if "Q_DIAS_STOCK" in stock_sucursal.columns:
                stock_sucursal["Q_DIAS_STOCK"] = pd.to_numeric(stock_sucursal["Q_DIAS_STOCK"], errors="coerce").astype("Int64")

            stock_sucursal.to_csv(archivo_stock, index=False, encoding="utf-8")
            print(f"---> Datos de Stock Sucursal guardados")

        # --- VENTAS (Diarco + Barrio) usando el mismo CTE ---
        if isinstance(ventana, (int, np.integer)) and ventana > 0:
            fecha_desde = (datetime.today() - timedelta(days=int(ventana))).date().isoformat()
        else:
            fecha_desde = "2024-06-01"

        params_ventas = dict(params)
        params_ventas["fecha_desde"] = fecha_desde

        sql_ventas = f"""
            {cte}
            SELECT 
                v.f_venta AS fecha,
                v.c_articulo AS codigo_articulo,
                v.c_sucu_empr AS sucursal,
                v.q_unidades_vendidas AS unidades
            FROM src.t702_est_vtas_por_articulo v
            JOIN articulos_filtrados af
            ON af.c_articulo = v.c_articulo AND af.c_sucu_empr = v.c_sucu_empr
            WHERE v.f_venta >= %(fecha_desde)s

            UNION ALL

            SELECT 
                v.f_venta AS fecha,
                v.c_articulo AS codigo_articulo,
                v.c_sucu_empr AS sucursal,
                v.q_unidades_vendidas AS unidades
            FROM src.t702_est_vtas_por_articulo_dbarrio v
            JOIN articulos_filtrados af
            ON af.c_articulo = v.c_articulo AND af.c_sucu_empr = v.c_sucu_empr
            WHERE v.f_venta >= %(fecha_desde)s

            ORDER BY fecha, codigo_articulo, sucursal;
            """.strip()

        ventas = pd.read_sql(sql_ventas, conn, params=params_ventas)  # type: ignore
        if ventas.empty:
            print(f"⚠️ No se encontraron ventas para proveedor={id_proveedor} desde {fecha_desde}.")
            ventas = pd.DataFrame(columns=["fecha","codigo_articulo","sucursal","unidades"])
        else:
            ventas.columns = ventas.columns.str.lower()
            ventas["sucursal"]        = pd.to_numeric(ventas["sucursal"], errors="coerce").astype("Int64")
            ventas["codigo_articulo"] = pd.to_numeric(ventas["codigo_articulo"], errors="coerce").astype("Int64")
            ventas["fecha"]           = pd.to_datetime(ventas["fecha"], errors="coerce")
            ventas = ventas.dropna(subset=["fecha","codigo_articulo","sucursal"], how="any")

        ventas = ventas.rename(columns={
            "fecha":"Fecha",
            "codigo_articulo":"Codigo_Articulo",
            "sucursal":"Sucursal",
            "unidades":"Unidades"
        })
        ventas.to_csv(archivo_demanda, index=False, encoding="utf-8")
        print(f"---> Datos de Ventas (compactos) guardados")

        # --- MERGE Artículos x Ventas ---
        data = pd.merge(
            articulos,
            ventas,
            left_on=["C_ARTICULO","C_SUCU_EMPR"],
            right_on=["Codigo_Articulo","Sucursal"],
            how="left",
        )

        for col in ["C_ARTICULO","C_SUCU_EMPR","Codigo_Articulo","Sucursal"]:
            if col in data.columns:
                data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

        data["TIENE_DEMANDA"] = data["Unidades"].notna().astype("Int64")
        data["Unidades"]      = pd.to_numeric(data["Unidades"], errors="coerce").fillna(0).astype("Float64")

        data.to_csv(archivo_datos, index=False, encoding="utf-8")
        print(f"---> Datos de RECUPERACIÓN guardados")

        # Export adicional sólo ventas para trazabilidad externa
        only_sales = data[["Fecha","Codigo_Articulo","Sucursal","Unidades"]].copy()
        only_sales.to_csv(archivo_ventas_out, index=False, encoding="utf-8")
        print(f"---> Datos de Ventas guardados: {archivo_ventas_out}")

        return data, articulos

    finally:
        Close_Connection(conn)


In [ ]:
from typing import Iterable, Optional, Tuple, Dict, Any
import datetime as dt


def parametros_de_gen_datos(
    id_proveedor: int,
    etiqueta: str,
    ventana: Optional[int],
    sucursales: Optional[Union[int, Iterable[int]]] = None,
    rubros: Optional[Union[int, Iterable[int]]] = None,
    subrubro1: Optional[Union[int, Iterable[int]]] = None,
    subrubro2: Optional[Union[int, Iterable[int]]] = None,
    subrubro3: Optional[Union[int, Iterable[int]]] = None,
):


In [ ]:
name = "20_ARCOR_SIN_RUBROS"
algoritmo = "ALGO_05"
id_proveedor = 20
etiqueta = f"{name}_{algoritmo}"
sucursales =  [1, 2, 49,302] # Ejemplo de sucursales a filtrar
#rubros = [2055, 2053]  # Ejemplo de rubros a filtrar
rubros = None
period_lengh = 45


data, articulos = generar_datos(id_proveedor, etiqueta, period_lengh, sucursales, rubros) # type: ignore
#get_forecast( id_proveedor, name, period_lengh=90, algorithm=algoritmo, f1=0.5, f2=0.3, f3=0.2, current_date=None, sucursales=sucursales, rubros=rubros)  


# VERSIÓN FINAL

In [ ]:
import os
import sqlalchemy
from sqlalchemy import text, bindparam
from sqlalchemy.engine import Connection
from sqlalchemy.dialects.postgresql import ARRAY, INTEGER
from contextlib import contextmanager

# Asumimos que 'secrets' ya existe con las claves
def Open_Connexa_Alquemy() -> Connection:
    DB_TYPE = "postgresql+psycopg2"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    print(f"[DEBUG] Conectando a PostgreSQL en {DB_HOST}:{DB_PORT}, Base de Datos: {DB_NAME}...")

    engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,               # evita conexiones muertas
        pool_size=5, max_overflow=10,     # tunning básico
        future=True
    )
    return engine.connect()

# print("Conexión a PostgreSQL con SQLAlchemy establecida.")
# print(f"[DEBUG] Host: {secrets['PGP_HOST']}, Port: {secrets['PGP_PORT']}, DB: {secrets['PGP_DB']}" )

# conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"

from typing import Iterable, Optional, Dict, Any
import datetime as dt


def Open_Diarco_Data_Alquemy() -> Connection:
    DB_TYPE = "postgresql+psycopg2"
    DB_USER = secrets['PG_USER']
    DB_PASS = secrets['PG_PASSWORD']
    DB_HOST = secrets['PG_HOST']
    DB_PORT = secrets['PG_PORT']
    DB_NAME = secrets['PG_DB']

    print(f"[DEBUG] Conectando a PostgreSQL en {DB_HOST}:{DB_PORT}, Base de Datos: {DB_NAME}...")

    engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,               # evita conexiones muertas
        pool_size=5, max_overflow=10,     # tunning básico
        future=True
    )
    return engine.connect()

print("Conexión a PostgreSQL con SQLAlchemy establecida.")
print(f"[DEBUG] Host: {secrets['PG_HOST']}, Port: {secrets['PG_PORT']}, DB: {secrets['PG_DB']}" )

conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"

from typing import Iterable, Optional, Dict, Any
import datetime as dt

def build_params(
    id_proveedor: int,
    desde: str | dt.date,
    sucursales: Optional[Iterable[int]] = None,
    rubros: Optional[Iterable[int]] = None
) -> Dict[str, Any]:
    def _arr_or_null(x):
        if x is None:
            return None
        x = list(x)
        return x if len(x) > 0 else None

    print(f"[DEBUG] Parámetros construidos: id_proveedor={id_proveedor}, desde={desde}, sucursales={sucursales}, rubros={rubros}")
    return {
        "id_proveedor": int(id_proveedor),
        "desde": desde,
        "sucursales": _arr_or_null(sucursales),  # p.ej. [1,2,3] o None
        "rubros": _arr_or_null(rubros)           # p.ej. [2055,2053] o None
    }

def get_forecast(id_proveedor, lbl_proveedor, period_lengh=30, algorithm='basic',
                 f1=None, f2=None, f3=None, current_date=None,
                 sucursales=None, rubros=None):
    """
    Genera la predicción de demanda según el algoritmo seleccionado.
    """
    print('Dentro del get_forecast LOCAL')
    print(f'FORECAST control: {id_proveedor} - {lbl_proveedor} - ventana: {period_lengh} - {algorithm} factores: {f1} - {f2} - {f3} ')
    if sucursales is not None:
        print(f'Filtrando por sucursales (array): {sucursales}')
    if rubros is not None:
        print(f'Filtrando por rubros (array): {rubros}')

    data, articulos = generar_datos(id_proveedor, lbl_proveedor, period_lengh, sucursales, rubros)  # ojo: ahora pasa arrays

    # Determinar la fecha base
    if data is not None and not data.empty:
        if current_date is None:
            current_date = data['Fecha'].max()
        else:
            current_date = pd.to_datetime(current_date)
        print(f'Fecha actual {current_date}')
    else:
        print("⚠️ No hay datos para determinar current_date.")
    return data, articulos

from sqlalchemy import text, bindparam
from sqlalchemy.dialects.postgresql import ARRAY, INTEGER
import pandas as pd
import numpy as np
import os
from datetime import datetime

def generar_datos(id_proveedor, etiqueta, ventana, sucursales, rubros):
    folder = secrets["FOLDER_DATOS"]
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'

    # ---------- CACHE ----------
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')
                data['Codigo_Articulo'] = pd.to_numeric(data['Codigo_Articulo'], errors='coerce').astype('Int64')
                data['Sucursal'] = pd.to_numeric(data['Sucursal'], errors='coerce').astype('Int64')
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')
                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # limpiar cache viejo
            try:
                os.remove(archivo_datos)
            except FileNotFoundError:
                pass
            if os.path.exists(archivo_articulos):
                try:
                    os.remove(archivo_articulos)
                except FileNotFoundError:
                    pass
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")

    print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")

    # Armado de parámetros comunes (arrays o NULL)
    sucursales = list(sucursales) if sucursales else None
    rubros     = list(rubros) if rubros else None

    with Open_Diarco_Data_Alquemy() as conn:
        # ---------------------------
        # 4.1) ARTÍCULOS (parametrizado)
        # ---------------------------
        sql_articulos = text("""
            SELECT DISTINCT P.c_sucu_empr, P.c_articulo, P.c_proveedor_primario, P.abastecimiento, P.cod_cd, P.habilitado,
                   P.cod_comprador AS c_comprador,
                   P.q_factor_compra, P.full_capacity_pallet, P.number_of_layers, P.number_of_boxes_per_layer
            FROM src.base_productos_vigentes P
            LEFT JOIN src.m_3_articulos A
                   ON P.c_articulo = A.c_articulo
            WHERE P.c_proveedor_primario = :id_proveedor
              AND (:sucursales IS NULL OR P.c_sucu_empr = ANY(:sucursales))
              AND (:rubros     IS NULL OR A.c_rubro     = ANY(:rubros))
              AND P.habilitado = 1
            ORDER BY P.c_articulo, P.c_sucu_empr
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        print("[DEBUG] Ejecutando query de artículos...")
        articulos = pd.read_sql(
            sql_articulos, conn,
            params={"id_proveedor": id_proveedor, "sucursales": sucursales, "rubros": rubros}
        )

        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)

        # Cast consistente a tipos enteros “nullable”
        for c in ['C_SUCU_EMPR','C_ARTICULO','C_PROVEEDOR_PRIMARIO','ABASTECIMIENTO','HABILITADO','C_COMPRADOR',
                  'Q_FACTOR_COMPRA','FULL_CAPACITY_PALLET','NUMBER_OF_LAYERS','NUMBER_OF_BOXES_PER_LAYER']:
            articulos[c] = pd.to_numeric(articulos[c], errors='coerce').astype('Int64')

        articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        print("---> Datos de Artículos guardados")

        # -------------------------------------
        # 4.2) STOCK POR SUCURSAL (parametrizado)
        # -------------------------------------
        sql_stock = text("""
            SELECT S.codigo_articulo, S.codigo_sucursal, S.codigo_proveedor, S.pedido_sgm, S.stock,
                   S.pedido_pendiente, S.i_lista_calculado, S.factor_venta, S.ultimo_ingreso, S.fecha_ultimo_ingreso,
                   S.fecha_ultima_venta, S.precio_venta, S.precio_costo,
                   S.q_dias_stock, S.transfer_pendiente, S.venta_unidades_1q, S.venta_unidades_2q
            FROM src.base_stock_sucursal S
            LEFT JOIN src.m_3_articulos A
                   ON S.codigo_articulo = A.c_articulo
            WHERE S.codigo_proveedor = :id_proveedor
              AND (:sucursales IS NULL OR S.codigo_sucursal = ANY(:sucursales))
              AND (:rubros     IS NULL OR A.c_rubro        = ANY(:rubros))
            ORDER BY S.codigo_articulo, S.codigo_sucursal
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        print("[DEBUG] Ejecutando query de stock_sucursal...")
        stock_sucursal = pd.read_sql(
            sql_stock, conn,
            params={"id_proveedor": id_proveedor, "sucursales": sucursales, "rubros": rubros}
        )

        if stock_sucursal.empty:
            print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
            return None, None

        stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan).fillna(0)
        stock_sucursal.columns = stock_sucursal.columns.str.upper()

        stock_sucursal['CODIGO_SUCURSAL'] = pd.to_numeric(stock_sucursal['CODIGO_SUCURSAL'], errors='coerce').astype('Int64')
        stock_sucursal['CODIGO_ARTICULO'] = pd.to_numeric(stock_sucursal['CODIGO_ARTICULO'], errors='coerce').astype('Int64')
        stock_sucursal['CODIGO_PROVEEDOR'] = pd.to_numeric(stock_sucursal['CODIGO_PROVEEDOR'], errors='coerce').astype('Int64')

        for c in ["PEDIDO_SGM","STOCK","PEDIDO_PENDIENTE","I_LISTA_CALCULADO","ULTIMO_INGRESO",
                  "PRECIO_VENTA","PRECIO_COSTO","TRANSFER_PENDIENTE","VENTA_UNIDADES_1Q","VENTA_UNIDADES_2Q"]:
            stock_sucursal[c] = pd.to_numeric(stock_sucursal[c], errors='coerce').astype('Float64')

        stock_sucursal['FACTOR_VENTA'] = pd.to_numeric(stock_sucursal['FACTOR_VENTA'], errors='coerce').astype('Int64')
        stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')

        stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        print("---> Datos de Stock Sucursal guardados")

        # ------------------------
        # 4.3) VENTAS (con CTEs)
        # ------------------------
        sql_ventas = text("""
            WITH ventas_unificadas AS (
                SELECT v.f_venta AS fecha,
                       v.c_articulo AS codigo_articulo,
                       v.c_sucu_empr AS sucursal,
                       v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo v
                WHERE v.f_venta >= :desde
                UNION ALL
                SELECT v.f_venta AS fecha,
                       v.c_articulo AS codigo_articulo,
                       v.c_sucu_empr AS sucursal,
                       v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                WHERE v.f_venta >= :desde
            ),
            articulos_filtrados AS (
                SELECT m.c_articulo
                FROM src.m_3_articulos m
                WHERE :rubros IS NULL OR m.c_rubro = ANY(:rubros)
            ),
            surtido_filtrado AS (
                SELECT a.c_articulo  AS codigo_articulo,
                       a.c_sucu_empr AS sucursal
                FROM src.base_productos_vigentes a
                WHERE a.c_proveedor_primario = :id_proveedor
                  AND (:sucursales IS NULL OR a.c_sucu_empr = ANY(:sucursales))
                  AND (:rubros IS NULL OR a.c_articulo IN (SELECT c_articulo FROM articulos_filtrados))
            )
            SELECT v.fecha, v.codigo_articulo, v.sucursal, v.unidades
            FROM ventas_unificadas v
            JOIN surtido_filtrado s
              ON s.codigo_articulo = v.codigo_articulo
             AND s.sucursal        = v.sucursal
            ORDER BY v.fecha, v.codigo_articulo, v.sucursal
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("desde"),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        params_ventas = build_params(
            id_proveedor=id_proveedor,
            desde="2024-06-01",
            sucursales=sucursales,
            rubros=rubros
        )

        print("[DEBUG] Ejecutando query de ventas...")
        ventas = pd.read_sql(sql_ventas, conn, params=params_ventas)

    # ------- fuera del with: merge y salidas -------
    if ventas.empty:
        print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        ventas = pd.DataFrame(columns=['fecha','codigo_articulo','sucursal','unidades'])

    if not ventas.empty:
        ventas.columns = ventas.columns.str.lower()
        ventas['sucursal'] = pd.to_numeric(ventas['sucursal'], errors='coerce').astype('Int64')
        ventas['codigo_articulo'] = pd.to_numeric(ventas['codigo_articulo'], errors='coerce').astype('Int64')
        ventas['fecha'] = pd.to_datetime(ventas['fecha'])
        ventas = ventas.dropna(subset=['fecha','codigo_articulo','sucursal'], how='all')

    ventas = ventas.rename(columns={
        "fecha": "Fecha",
        "codigo_articulo": "Codigo_Articulo",
        "sucursal": "Sucursal",
        "unidades": "Unidades"
    })
    ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
    print("---> Datos de Ventas guardados")

    # --- MERGE FINAL ---
    data = pd.merge(
        articulos,
        ventas,
        left_on=['C_ARTICULO', 'C_SUCU_EMPR'],
        right_on=['Codigo_Articulo', 'Sucursal'],
        how='left'
    )

    if data.empty:
        print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
        return None, articulos

    # Casting seguro para columnas clave
    for c in ['C_ARTICULO', 'C_SUCU_EMPR', 'Codigo_Articulo', 'Sucursal']:
        data[c] = pd.to_numeric(data[c], errors='coerce').astype('Int64')

    data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
    data['Unidades'] = pd.to_numeric(data['Unidades'], errors='coerce').fillna(0)

    data.to_csv(archivo_datos, index=False, encoding='utf-8')
    print("---> Datos de RECUPERACIÓN guardados")

    # Compacto de ventas
    file_path = f'{folder}/{etiqueta}_Ventas.csv'
    print(f"[DEBUG] Ruta destino definida en .env: {folder}")
    ventas_compacto = data[['Fecha','Codigo_Articulo','Sucursal','Unidades']]
    ventas_compacto.to_csv(file_path, index=False, encoding='utf-8')
    print(f"---> Datos de Ventas guardados: {file_path}")

    return data, articulos


In [ ]:
name = "20_ARCOR"
algoritmo = "ALGO_05"
id_proveedor = 20
sucursales = [1, 2, 49, 302]   # listas (arrays)
rubros = [2055, 2053]          # listas (arrays)

data, articulos = get_forecast(
    id_proveedor, name, period_lengh=90,
    algorithm=algoritmo, f1=0.5, f2=0.3, f3=0.2,
    current_date=None,
    sucursales=sucursales, rubros=rubros
)


# PRUEBA de CONEXIONES

In [ ]:
# diag_conexion_tabla.py
# Verifica conexión y existencia/permiso de una tabla con los mismos parámetros de su .env
# Requisitos: pip install python-dotenv sqlalchemy psycopg2-binary pandas

import os
from typing import Dict, Any, Tuple, Optional

import pandas as pd
from dotenv import dotenv_values

import psycopg2
from psycopg2.extras import RealDictCursor

import sqlalchemy
from sqlalchemy import text
from sqlalchemy.engine import URL


def load_secrets_from_env() -> Dict[str, Any]:
    """Carga el .env igual que su código productivo."""
    env_path = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")
    secrets = dotenv_values(env_path)
    required = ["PGP_HOST", "PGP_PORT", "PGP_DB", "PGP_USER", "PGP_PASSWORD"]
    missing = [k for k in required if not secrets.get(k)]
    if missing:
        raise RuntimeError(f"Faltan claves en {env_path}: {missing}")
    return secrets


def build_sqlalchemy_engine(secrets: Dict[str, Any]) -> sqlalchemy.Engine:
    """
    Construye el Engine asegurando el ESCAPADO de credenciales.
    Usamos el mismo host/port/db/user/password que psycopg2.
    """
    url = URL.create(
        drivername="postgresql+psycopg2",   # driver explícito y estable
        username=secrets["PG_USER"],
        password=secrets["PG_PASSWORD"],   # URL.create hace el quoting correcto
        host=secrets["PG_HOST"],
        port=int(secrets["PG_PORT"]),
        database=secrets["PG_DB"],
    )
    return sqlalchemy.create_engine(url, pool_pre_ping=True, future=True)


def build_psycopg2_conn(secrets: Dict[str, Any]) -> psycopg2.extensions.connection:
    """Crea la conexión psycopg2 con los mismos parámetros del .env."""
    return psycopg2.connect(
        host=secrets["PG_HOST"],
        port=int(secrets["PG_PORT"]),
        dbname=secrets["PG_DB"],
        user=secrets["PG_USER"],
        password=secrets["PG_PASSWORD"],
        cursor_factory=RealDictCursor,
    )


def _split_fqname(fqname: str) -> Tuple[str, str]:
    """
    'src.base_productos_vigentes' -> ('src', 'base_productos_vigentes')
    """
    parts = fqname.split(".")
    if len(parts) != 2:
        raise ValueError(f"Nombre de tabla no calificado: {fqname} (esperado esquema.tabla)")
    return parts[0], parts[1]


def fmt_row(row: Optional[dict]) -> str:
    return "NULL" if row is None else ", ".join(f"{k}={v}" for k, v in row.items())


WHOAMI_SQL = """
SELECT
  current_database()           AS db,
  current_user                 AS user,
  session_user                 AS session_user,
  inet_server_addr()::text     AS server_ip,
  inet_server_port()           AS server_port,
  version()                    AS version;
"""

SEARCH_PATH_SQL = "SHOW search_path;"

EXISTENCE_SQL = """
SELECT
  to_regclass(:fqname)                                           AS regclass,
  EXISTS (
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema = :schema AND table_name = :table
  )                                                              AS in_information_schema,
  (SELECT has_schema_privilege(current_user, :schema, 'USAGE'))  AS has_schema_usage,
  (SELECT has_table_privilege(current_user, :fqname, 'SELECT'))  AS can_select
;
"""

def probe_sql(schema: str, table: str) -> str:
    return f"SELECT 1 FROM {schema}.{table} LIMIT 1;"


def verificar_sqlalchemy(secrets: Dict[str, Any], fq_table: str) -> None:
    schema, table = _split_fqname(fq_table)
    print("\n=== Verificación con SQLAlchemy ===")
    engine = build_sqlalchemy_engine(secrets)
    with engine.connect() as conn:
        who = conn.execute(text(WHOAMI_SQL)).mappings().first()
        print(f"[WHOAMI] {fmt_row(dict(who) if who else None)}")

        sp = conn.execute(text(SEARCH_PATH_SQL)).scalar()
        print(f"[search_path] {sp}")

        exist = conn.execute(
            text(EXISTENCE_SQL),
            {"fqname": fq_table, "schema": schema, "table": table},
        ).mappings().first()
        print(f"[existencia] {fmt_row(dict(exist) if exist else None)}")

        # Prueba de acceso real
        try:
            conn.execute(text(probe_sql(schema, table)))
            print("[probe] SELECT 1 ... OK")
        except Exception as e:
            print(f"[probe] ERROR -> {type(e).__name__}: {e}")


def verificar_psycopg2(secrets: Dict[str, Any], fq_table: str) -> None:
    schema, table = _split_fqname(fq_table)
    print("\n=== Verificación con psycopg2 ===")
    with build_psycopg2_conn(secrets) as conn:
        with conn.cursor() as cur:
            cur.execute(WHOAMI_SQL)
            who = cur.fetchone()
            print(f"[WHOAMI] {fmt_row(who)}")

            cur.execute(SEARCH_PATH_SQL)
            sp = cur.fetchone()
            print(f"[search_path] {sp.get('search_path') if sp else 'NULL'}")

            cur.execute(
                """
                SELECT
                  to_regclass(%s) AS regclass,
                  EXISTS (
                    SELECT 1 FROM information_schema.tables
                    WHERE table_schema = %s AND table_name = %s
                  ) AS in_information_schema,
                  has_schema_privilege(current_user, %s, 'USAGE') AS has_schema_usage,
                  has_table_privilege(current_user, %s, 'SELECT') AS can_select
                """,
                (fq_table, schema, table, schema, fq_table)
            )
            exist = cur.fetchone()
            print(f"[existencia] {fmt_row(exist)}")

            try:
                cur.execute(probe_sql(schema, table))
                _ = cur.fetchone()
                print("[probe] SELECT 1 ... OK")
            except Exception as e:
                print(f"[probe] ERROR -> {type(e).__name__}: {e}")


def diagnostico(fq_table: str = "src.base_productos_vigentes") -> None:
    """Ejecuta la verificación con ambos conectores usando el .env de producción."""
    secrets = load_secrets_from_env()

    # Mostrar parámetros efectivos (sin password)
    print("=== Parámetros de conexión cargados del .env ===")
    print(f"Host={secrets['PGP_HOST']}  Port={secrets['PGP_PORT']}  DB={secrets['PGP_DB']}  User={secrets['PGP_USER']}")
    print(f"Tabla objetivo: {fq_table}")

    try:
        verificar_sqlalchemy(secrets, fq_table)
    except Exception as e:
        print(f"\n[SQLAlchemy] ERROR -> {type(e).__name__}: {e}")

    try:
        verificar_psycopg2(secrets, fq_table)
    except Exception as e:
        print(f"\n[psycopg2] ERROR -> {type(e).__name__}: {e}")

    print("\n=== Pistas de diagnóstico si difieren ===")
    print("- Si [WHOAMI] o server_ip/server_port difieren entre conectores, están apuntando a servidores distintos.")
    print("- Si 'regclass' es NULL pero 'in_information_schema' es True, podría haber tema de comillas/caso del nombre.")
    print("- Si 'has_schema_usage' = False, falta USAGE sobre el esquema (p.ej. 'src').")
    print("- Si 'can_select' = False, falta SELECT sobre la tabla.")
    print("- Si el probe falla solo en SQLAlchemy, revisen contraseñas con caracteres especiales: usen URL.create (como aquí).")
    print("- Si todo está OK en psycopg2 y falla en SQLAlchemy, comparen SEARCH_PATH; a veces hay GUCs aplicadas por cliente.")



In [ ]:
# if __name__ == "__main__":
#     # Ejemplo de uso:
#     try:
#         from secrets_local import secrets  # o importen su dict real
#     except Exception:
#         print("⚠️ Deben proveer el dict 'secrets' con PGP_HOST/PORT/DB/USER/PASSWORD.")
#         sys.exit(1)

diagnostico( fq_table="src.base_productos_vigentes")

## INTEGRACIÓN CONTINUA (CI/CD) con GITLAB

In [ ]:
# Selección del algoritmo de predicción
match algorithm:
    case 'ALGO_01':
        return Procesar_ALGO_01(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, factor_last=f1, factor_previous=f2, factor_year=f3)  # Promedio Ponderado x 3 Factores
    case 'ALGO_02':
        return Procesar_ALGO_02(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Doble Exponencial - Modelo Holt (Tendencia)
    case 'ALGO_03':
        return Procesar_ALGO_03(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, periodos=f1, estacionalidad=f2, tendencia=f3) # Triple Exponencial Holt-WInter (Tendencia + Estacionalidad) (periodos, add, add)
    case 'ALGO_04':
        return Procesar_ALGO_04(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, alpha=f1) # EWMA con Factor alpha
    case 'ALGO_05':
        return Procesar_ALGO_05(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Promedio Venta Simple en Ventana
    case 'ALGO_06':
        return Procesar_ALGO_06(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Tendencias Ventas Semanales
    case 'ALGO_07':
        return Procesar_ALGO_07(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, factor=f1, fecha_base=f2)  # Promedio Simple Ventana Base Movil x Factor
    case _:
        raise ValueError(f"Error: El algoritmo '{algorithm}' no está implementado.")
